In [5]:
"""
=============================================================================
PHASE 1: DATA PREPARATION & FEATURE ENGINEERING
Project: Decoding Customer Value — A SQL-Driven Retention Strategy
Author:  Senior Analytics Consultant Framework
=============================================================================

ANALYTICAL PHILOSOPHY
─────────────────────
Every feature built here must answer a question the brand actually cares about.
We do not engineer metrics for completeness — we engineer them for decisions.
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: LOAD & AUDIT
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv("Dataset.csv")

print("=" * 65)
print("STEP 1 — DATA AUDIT")
print("=" * 65)
print(f"Total customers : {len(df):,}")
print(f"Total columns   : {len(df.columns)}")
print(f"Missing values  :\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: CLEANING
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2 — CLEANING")
print("=" * 65)

# Handle missing Review Ratings safely using median
median_rating = df['Review Rating'].median()
df['Review Rating'] = df['Review Rating'].fillna(median_rating)
print(f"Review Rating nulls filled with median: {median_rating}")

# Standardise string columns safely
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

print(f"String columns standardised: {list(str_cols)}")
print(f"No duplicate Customer IDs: {df['Customer ID'].nunique() == len(df)}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: BINARY ENCODINGS
# ─────────────────────────────────────────────────────────────────────────────

df['promo_flag'] = (df['Discount Applied'] == 'Yes').astype(int)
df['subscribed'] = (df['Subscription Status'] == 'Yes').astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 4 — FEATURE ENGINEERING")
print("=" * 65)

# ── FEATURE 1: Frequency Score ──────────────────────────────────────────────
frequency_map = {
    'Weekly':          7,
    'Bi-Weekly':       6,
    'Fortnightly':     5,
    'Monthly':         4,
    'Every 3 Months':  3,
    'Quarterly':       2,
    'Annually':        1
}
df['frequency_score'] = df['Frequency of Purchases'].map(frequency_map)
# Fallback for unexpected string variations or missing values in frequency mapping
df['frequency_score'] = df['frequency_score'].fillna(1).astype(int)

print("✓ Feature 1: frequency_score (1–7 ordinal, 7 = most frequent)")

# ── FEATURE 2: Behavioral Loyalty Index (BLI) ───────────────────────────────
df['prev_purchases_norm'] = df['Previous Purchases'] / df['Previous Purchases'].max()
df['freq_score_norm']     = (df['frequency_score'] - 1) / 6   # scale 1–7 → 0–1

df['BLI'] = df['prev_purchases_norm'] * df['freq_score_norm']

print("✓ Feature 2: BLI — Behavioral Loyalty Index (0–1)")

# ── FEATURE 3: Intrinsic Loyalty Score (ILS) ────────────────────────────────
df['satisfied_flag']  = (df['Review Rating'] >= 4.0).astype(int)
df['organic_buyer']   = (df['promo_flag'] == 0).astype(int)

df['ILS'] = df['subscribed'] + df['satisfied_flag'] + df['organic_buyer']

print("✓ Feature 3: ILS — Intrinsic Loyalty Score (0–3)")

# ── FEATURE 4: Estimated Lifetime Value (eLTV) ──────────────────────────────
df['eLTV'] = df['Purchase Amount (USD)'] * df['Previous Purchases']

print("✓ Feature 4: eLTV — Estimated Lifetime Value ($)")

# ── FEATURE 5: Customer Value Tier ──────────────────────────────────────────
tertile_low  = df['eLTV'].quantile(0.33)
tertile_high = df['eLTV'].quantile(0.67)

def assign_value_tier(eltv):
    if eltv >= tertile_high:
        return 'High'
    elif eltv >= tertile_low:
        return 'Mid'
    else:
        return 'Low'

df['value_tier'] = df['eLTV'].apply(assign_value_tier)

print("✓ Feature 5: value_tier (High / Mid / Low) based on eLTV tertiles")

# ── FEATURE 6: Satisfaction Flag ────────────────────────────────────────────
def satisfaction_label(r):
    if r >= 4.0:   return 'Satisfied'
    elif r >= 3.0: return 'Neutral'
    else:          return 'Dissatisfied'

df['satisfaction_flag'] = df['Review Rating'].apply(satisfaction_label)

print("✓ Feature 6: satisfaction_flag (Satisfied / Neutral / Dissatisfied)")

# ── FEATURE 7: Promo Dependency Label ───────────────────────────────────────
df['promo_dependency'] = df['promo_flag'].map({1: 'Promo-Dependent', 0: 'Organic'})

print("✓ Feature 7: promo_dependency (Promo-Dependent / Organic)")

# ── FEATURE 8: Customer Segment Label ───────────────────────────────────────
# CRITICAL FIX: Safe qcut execution using duplicates='drop' to avoid edge collisions
try:
    df['BLI_quartile'] = pd.qcut(df['BLI'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
except ValueError:
    # Fallback to auto-generated labels if duplicate values merge bin boundaries
    df['BLI_quartile'] = pd.qcut(df['BLI'], q=4, duplicates='drop')

def assign_segment(row):
    high_value   = row['value_tier'] == 'High'
    # Ensure string check works regardless of whether qcut dropped categories
    high_loyalty = str(row['BLI_quartile']) in ['Q3', 'Q4'] or 'Q3' in str(row['BLI_quartile']) or 'Q4' in str(row['BLI_quartile'])
    organic      = row['promo_dependency'] == 'Organic'
    is_subscribed = row['subscribed'] == 1

    # 1. High-Value Loyal: High spend, high tenure/frequency, buys without discount
    if high_value and high_loyalty and organic:
        return 'High-Value Loyal'

    # 2. Premium Subscriber: Subscribed, high value — but discount-dependent
    if is_subscribed and high_value:
        return 'Discount-Dependent Premium'

    # 3. Potential Loyalist: High loyalty behavior but not yet high spend
    if high_loyalty and organic and not high_value:
        return 'Potential Loyalist'

    # 4. Discount Hunter: Uses promos, lower tenure/frequency
    if not organic and not high_loyalty:
        return 'Discount Hunter'

    # 5. At-Risk: Low value, dissatisfied or neutral, low loyalty
    if row['value_tier'] == 'Low' and row['satisfaction_flag'] in ['Dissatisfied', 'Neutral']:
        return 'At-Risk'

    # 6. Passive / Occasional: Everyone else
    return 'Passive / Occasional'

df['customer_segment'] = df.apply(assign_segment, axis=1)

print("✓ Feature 8: customer_segment (6-tier strategic segmentation)")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: VALIDATION CHECKS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 5 — VALIDATION: Testing Both Loyalty Definitions")
print("=" * 65)

print("\n[BLI Validation] BLI quartile vs. Avg Purchase Amount:")
print(df.groupby('BLI_quartile', observed=False)['Purchase Amount (USD)'].mean().round(2))

print("\n[BLI Validation] BLI quartile vs. Avg Previous Purchases:")
print(df.groupby('BLI_quartile', observed=False)['Previous Purchases'].mean().round(2))

print("\n[ILS Validation] ILS score vs. Avg Previous Purchases:")
print(df.groupby('ILS')['Previous Purchases'].mean().round(2))

print("\n[ILS Validation] ILS score vs. Avg Purchase Amount:")
print(df.groupby('ILS')['Purchase Amount (USD)'].mean().round(2))

print("\n[Segment Counts]:")
print(df['customer_segment'].value_counts())

print("\n[Segment eLTV Summary]:")
print(df.groupby('customer_segment')['eLTV'].agg(['mean', 'median', 'count']).round(0))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: EXPORT CLEAN DATASET
# ─────────────────────────────────────────────────────────────────────────────
output_path = 'Datasetnew.csv'
df.to_csv(output_path, index=False)

print("\n" + "=" * 65)
print("STEP 6 — OUTPUT")
print("=" * 65)
print(f"✓ Cleaned + engineered dataset saved to:\n  {output_path}")
print(f"Total rows: {len(df)}, Total columns: {len(df.columns)}")
print(f"\nNew features added:")
new_features = ['promo_flag', 'subscribed', 'frequency_score', 'BLI', 'ILS',
                'eLTV', 'value_tier', 'satisfaction_flag', 'promo_dependency',
                'BLI_quartile', 'customer_segment', 'prev_purchases_norm',
                'freq_score_norm', 'satisfied_flag', 'organic_buyer']
for f in new_features:
    print(f"  + {f}")

STEP 1 — DATA AUDIT
Total customers : 3,900
Total columns   : 18
Missing values  :
Review Rating    37
dtype: int64

STEP 2 — CLEANING
Review Rating nulls filled with median: 3.8
String columns standardised: ['Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Payment Method', 'Frequency of Purchases']
No duplicate Customer IDs: True

STEP 4 — FEATURE ENGINEERING
✓ Feature 1: frequency_score (1–7 ordinal, 7 = most frequent)
✓ Feature 2: BLI — Behavioral Loyalty Index (0–1)
✓ Feature 3: ILS — Intrinsic Loyalty Score (0–3)
✓ Feature 4: eLTV — Estimated Lifetime Value ($)
✓ Feature 5: value_tier (High / Mid / Low) based on eLTV tertiles
✓ Feature 6: satisfaction_flag (Satisfied / Neutral / Dissatisfied)
✓ Feature 7: promo_dependency (Promo-Dependent / Organic)
✓ Feature 8: customer_segment (6-tier strategic segmentation)

STEP 5 — VALIDATION: Testing Both Loyalty Definitions

[BLI Va